[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yildiztu/Code-101-Problems-Logistics-SCM/blob/main/Part%20I%20-%20The%20Port%20%26%20Container%20Terminal%20%28Problems%201-46%29/C.%20Core%20Yard%20%26%20Land-Side%20Operations%20%28The%20Heart%20of%20the%20Terminal%29/23.%20The%20AGV%20Dispatching%20%26%20Battery%20Management%20Problem/P23-Tier-3_executed.ipynb) [![Open in SageMaker](https://img.shields.io/badge/Open%20in-SageMaker-orange?logo=amazon-aws)](https://aws.amazon.com/sagemaker/) [![Open in Azure](https://img.shields.io/badge/Open%20in-Azure%20Notebooks-blue?logo=microsoft-azure)](https://notebooks.azure.com/)

---


# Problem 23: The AGV Dispatching and Battery Management Problem

## Tier 3: Ant Colony Optimization for AGV Dispatching

### Goal and workflow

This notebook uses Ant Colony Optimization (ACO), a population-based metaheuristic, to construct and improve routes for automated guided vehicles (AGVs). Each ant builds a candidate solution by making probabilistic routing choices. Pheromone trails record information from feasible, low-cost solutions and influence later choices. ACO is a search method: it can find good solutions, but it does not certify that a solution is globally optimal.

The notebook proceeds through the following steps:

1. Represent the depot, task pickup and drop-off locations, and charging stations as nodes in a directed travel graph.
2. Initialize pheromone trails and distance-based heuristic values for possible movements.
3. Assign each task to an AGV and construct each AGV's route probabilistically, respecting pickup-before-drop-off precedence.
4. Evaluate route travel time and battery feasibility, including the return to the depot.
5. Evaporate pheromone and reinforce edges used by feasible solutions.
6. Repeat the construction and update steps, recording solution quality and population diversity.

### Model assumptions and scope

- Every task is assigned to exactly one AGV, and its pickup must occur before its drop-off. Each task is treated as an indivisible pickup-and-delivery job.
- AGVs start at the depot and return there. Travel times and energy consumption are symmetric functions of Euclidean distance; the supplied travel-time matrix uses a common speed of one distance unit per time unit.
- A route may visit each charging station at most once. On arrival, the AGV is assumed to recharge immediately to full capacity. Charging duration and station capacity are not modeled.
- Battery feasibility means the battery level does not fall below the AGV's minimum reserve after a travel leg. The trip to a charging station must itself satisfy this reserve.
- The data include time windows, deadlines, service times, task priorities, AGV-specific speeds, and charging rates, but this ACO model does not use those fields. Its objective is total travel time, not a weighted or multi-objective cost.
- Battery awareness influences the probability of choosing the next node, while feasibility is checked after a complete route is built. Infeasible solutions do not deposit pheromone. This is not a penalty added to the travel-time objective.

### What to examine

The results report the best feasible solution found, the number of feasible ants per generation, the average cost among feasible ants, and a route-set diversity measure. These are empirical properties of this seeded run, not guarantees about all instances or proof of optimality. The simple greedy benchmark later in the notebook is included only as a reference for this example.

The example uses six tasks, three AGVs, two charging stations, 20 ants, and 100 generations. The data and algorithm parameters are defined in the code cells below.

## ACO model and notation

### Routes, feasibility, and objective

Let $K$ be the set of AGVs, $T$ the set of tasks, $C$ the set of charging stations, and $V$ the set of locations. For each AGV $k \in K$, a route $R_k=(v_{k,0},v_{k,1},\ldots,v_{k,m_k})$ starts and ends at depot $O$. Every task $t \in T$ is assigned to one AGV, and its pickup node $P_t$ must appear before its drop-off node $D_t$. Charging nodes may appear between task nodes; each station can appear at most once in a route.

The cost minimized by the algorithm is the sum of travel times across all AGV routes:

$$
C(R)=\sum_{k\in K}\sum_{i=0}^{m_k-1} c_{v_{k,i},v_{k,i+1}},
$$

where $c_{ij}$ is the travel time from location $i$ to location $j$. A solution is feasible when all tasks are covered exactly once, pickup precedes the corresponding drop-off, and each AGV maintains at least its minimum battery reserve after every travel leg. A charging stop restores the battery to capacity after the arrival leg has passed the reserve check. The implementation evaluates battery feasibility on each complete route, including the final return to the depot.

### Probabilistic route construction

For a candidate move from node $i$ to node $j$, the unnormalized selection weight is

$$
w_{ij}=\tau_{ij}^{\alpha}\eta_{ij}^{\beta}b_{ij},
\qquad
p_{ij}=\frac{w_{ij}}{\sum_{\ell\in A}w_{i\ell}},
$$

where $\tau_{ij}$ is the pheromone level, $\eta_{ij}$ is heuristic attractiveness, $\alpha$ and $\beta$ control their relative influence, and $A$ is the set of currently eligible next nodes. The heuristic is the reciprocal of travel time plus a small energy term. The battery factor $b_{ij}$ reduces the weight of moves predicted to cross the reserve threshold and increases the weight of a charging-station move in that situation. This factor guides construction; it does not make an infeasible route feasible.

Eligibility enforces task precedence: a task's pickup is eligible until visited, and its drop-off becomes eligible only after that pickup. Unvisited charging stations are also candidates. A roulette-wheel draw selects one candidate according to the normalized probabilities. Thus, each ant constructs a complete route set using local probabilistic choices rather than enumerating all possible solutions.

### Pheromone update and iteration

After all ants have constructed solutions, each pheromone trail evaporates and feasible routes deposit pheromone:

$$
\tau_{ij}\leftarrow(1-\rho)\tau_{ij}
+\sum_{s\in F}\frac{q}{C(R^{(s)})}\,n_{ij}(R^{(s)}),
$$

where $\rho$ is the evaporation rate, $q$ is the deposition constant, $F$ is the set of feasible ant solutions in the generation, and $n_{ij}(R^{(s)})$ counts how many times solution $s$ uses directed edge $(i,j)$. In this implementation, routes do not repeat task or charging nodes, although a depot edge can occur in different AGV routes. Lower-cost feasible solutions deposit more pheromone. Evaporation limits the persistence of old information and allows later choices to reflect newer solutions.

The algorithm retains the best feasible solution found across generations. With $G$ generations, $A$ ants, and $N$ graph nodes, route construction and route evaluation are approximately $O(GAN^2)$ in the worst case. Pairwise population-diversity calculation adds approximately $O(GA^2|K|N)$. The pheromone and heuristic tables each require $O(N^2)$ storage. These bounds describe this implementation's main loops; actual work depends on the number of task and charging nodes visited.

In the code, $\tau_{ij}$ is `self.pheromone[(i, j)]`, $\eta_{ij}$ is `self.heuristic[(i, j)]`, $b_{ij}$ is computed in `calculate_transition_probability`, and the update is implemented by `update_pheromones`. `construct_ant_solution`, `build_agv_route`, and `evaluate_route` implement assignment, route construction, and feasibility/cost evaluation, respectively.

In [ ]:
# Import the libraries used by the ACO implementation and its visualizations.
import random
import time
from dataclasses import dataclass
from typing import Dict, List, Tuple

import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns

# Fixed seeds make the stochastic construction reproducible in a given environment.
np.random.seed(42)
random.seed(42)

plt.style.use("default")
sns.set_palette("husl")

In [ ]:
@dataclass
class Task:
    """A transport task with pickup, delivery, and scheduling attributes."""

    id: str
    pickup: str
    dropoff: str
    time_window: Tuple[float, float]
    service_time: float
    priority: float = 1.0
    deadline: float = float("inf")


@dataclass
class AGV:
    """AGV specifications; this model uses the battery fields."""

    id: str
    battery_capacity: float
    battery_min: float
    initial_battery: float
    speed: float = 1.0


@dataclass
class ChargingStation:
    """Charging-station attributes; charging duration is out of scope."""

    id: str
    charging_rate: float


@dataclass
class Location:
    """A terminal location represented by planar coordinates."""

    id: str
    x: float
    y: float

    def distance_to(self, other: "Location") -> float:
        """Return Euclidean distance, the geometry used for travel data."""
        return float(np.hypot(self.x - other.x, self.y - other.y))


@dataclass
class AntSolution:
    """Routes and evaluation metrics for one ant's candidate solution."""

    routes: Dict[str, List[str]]
    cost: float
    feasible: bool
    battery_violations: int
    time_violations: int


@dataclass
class PheromoneInfo:
    """Pheromone trail values and their update parameters."""

    trails: Dict[Tuple[str, str], float]
    evaporation_rate: float = 0.1
    deposition_constant: float = 1.0

In [ ]:
# Define the terminal locations used by the example instance.
locations = {
    "O": Location("O", 0, 0),
    "P1": Location("P1", 2, 3),
    "D1": Location("D1", 6, 4),
    "P2": Location("P2", 1, 5),
    "D2": Location("D2", 5, 7),
    "P3": Location("P3", 3, 1),
    "D3": Location("D3", 7, 2),
    "P4": Location("P4", 4, 6),
    "D4": Location("D4", 8, 5),
    "P5": Location("P5", 2, 7),
    "D5": Location("D5", 6, 8),
    "P6": Location("P6", 5, 3),
    "D6": Location("D6", 9, 4),
    "C1": Location("C1", 3, 3),
    "C2": Location("C2", 6, 1),
}

# Task scheduling attributes are retained as data, but are outside this model's scope.
tasks = [
    Task("T1", "P1", "D1", (0, 30), 2.0, priority=1.0, deadline=35),
    Task("T2", "P2", "D2", (5, 25), 1.5, priority=0.8, deadline=30),
    Task("T3", "P3", "D3", (10, 40), 2.5, priority=1.2, deadline=45),
    Task("T4", "P4", "D4", (15, 35), 1.8, priority=0.9, deadline=40),
    Task("T5", "P5", "D5", (8, 32), 2.2, priority=1.1, deadline=38),
    Task("T6", "P6", "D6", (12, 42), 1.6, priority=0.7, deadline=48),
]

agvs = [
    AGV("K1", 100.0, 20.0, 100.0, speed=1.0),
    AGV("K2", 120.0, 25.0, 120.0, speed=1.2),
    AGV("K3", 90.0, 15.0, 90.0, speed=0.9),
]

charging_stations = [
    ChargingStation("C1", 8.0),
    ChargingStation("C2", 12.0),
]


def calculate_matrices(locations):
    """Build symmetric travel-time and energy matrices from Euclidean distance."""
    travel_times = {}
    energy_consumption = {}

    for loc1_id, loc1 in locations.items():
        for loc2_id, loc2 in locations.items():
            distance = loc1.distance_to(loc2)
            travel_times[(loc1_id, loc2_id)] = distance
            energy_consumption[(loc1_id, loc2_id)] = 1.2 * distance

    return travel_times, energy_consumption


travel_times, energy_consumption = calculate_matrices(locations)

print("ACO problem setup:")
print(f"Tasks: {len(tasks)}")
print(f"AGVs: {len(agvs)}")
print(f"Charging stations: {len(charging_stations)}")
print(f"Locations: {list(locations)}")

print("\nTask attributes:")
for task in tasks:
    print(
        f"  {task.id}: priority={task.priority:.1f}, "
        f"deadline={task.deadline}, time window={task.time_window}"
    )

In [ ]:
class AGVAntColonyOptimizer:
    """Construct AGV routes with ACO and evaluate battery feasibility."""

    def __init__(
        self,
        locations,
        travel_times,
        energy_consumption,
        charging_stations,
        agvs,
        tasks,
        num_ants=20,
        generations=100,
        alpha=1.0,
        beta=2.0,
        rho=0.1,
        q=1.0,
    ):
        self.locations = locations
        self.travel_times = travel_times
        self.energy_consumption = energy_consumption
        self.charging_stations = charging_stations
        self.agvs = agvs
        self.tasks = tasks

        self.num_ants = num_ants
        self.generations = generations
        self.alpha = alpha
        self.beta = beta
        self.rho = rho
        self.q = q

        self.nodes = ["O"]
        for task in tasks:
            self.nodes.extend([task.pickup, task.dropoff])
        self.nodes.extend(station.id for station in charging_stations)

        self.pheromone = self.initialize_pheromones()
        self.heuristic = self.calculate_heuristic_info()

        self.best_solution = None
        self.best_cost = float("inf")
        self.convergence_history = []
        self.diversity_history = []

    def initialize_pheromones(self):
        """Give every directed non-self edge the same initial trail level."""
        initial_level = 1.0 / len(self.nodes)
        return {
            (i, j): initial_level
            for i in self.nodes
            for j in self.nodes
            if i != j
        }

    def calculate_heuristic_info(self):
        """Favor edges with lower travel-time-plus-energy heuristic cost."""
        heuristic = {}
        for i in self.nodes:
            for j in self.nodes:
                if i == j:
                    continue
                travel_time = self.travel_times.get((i, j), float("inf"))
                energy = self.energy_consumption.get((i, j), float("inf"))
                if np.isfinite(travel_time) and np.isfinite(energy):
                    heuristic[(i, j)] = 1.0 / (travel_time + 0.1 * energy)
                else:
                    heuristic[(i, j)] = 0.0
        return heuristic

    def calculate_transition_probability(
        self, current_node, next_node, visited_nodes, agv, current_battery
    ):
        """Return an unnormalized ACO weight for an eligible move."""
        if next_node in visited_nodes:
            return 0.0

        tau = self.pheromone.get((current_node, next_node), 0.0)
        eta = self.heuristic.get((current_node, next_node), 0.0)
        energy_needed = self.energy_consumption.get(
            (current_node, next_node), float("inf")
        )

        battery_factor = 1.0
        if current_battery - energy_needed < agv.battery_min:
            battery_factor = 0.1
            if next_node in {station.id for station in self.charging_stations}:
                battery_factor = 5.0

        return (tau**self.alpha) * (eta**self.beta) * battery_factor

    def construct_ant_solution(self, ant_id):
        """Randomly assign tasks, then construct one route per AGV."""
        routes = {agv.id: ["O"] for agv in self.agvs}
        task_assignments = {
            task.id: random.choice(self.agvs).id for task in self.tasks
        }

        total_cost = 0.0
        total_violations = 0
        for agv in self.agvs:
            assigned_tasks = [
                task
                for task in self.tasks
                if task_assignments[task.id] == agv.id
            ]
            route = self.build_agv_route(agv, assigned_tasks)
            routes[agv.id] = route
            route_cost, violations = self.evaluate_route(route, agv)
            total_cost += route_cost
            total_violations += violations

        return AntSolution(
            routes=routes,
            cost=total_cost,
            feasible=(total_violations == 0),
            battery_violations=total_violations,
            time_violations=0,
        )

    def build_agv_route(self, agv, tasks):
        """Build a precedence-respecting route with optional charging stops."""
        route = ["O"]
        current_node = "O"
        current_battery = agv.initial_battery
        visited_nodes = {"O"}
        picked_up_tasks = set()
        remaining_tasks = {task.id: task for task in tasks}

        while remaining_tasks:
            # A drop-off becomes eligible only after its own pickup is visited.
            eligible_task_nodes = []
            for task in remaining_tasks.values():
                if task.id in picked_up_tasks:
                    eligible_task_nodes.append(task.dropoff)
                else:
                    eligible_task_nodes.append(task.pickup)

            candidates = list(eligible_task_nodes)
            candidates.extend(
                station.id
                for station in self.charging_stations
                if station.id not in visited_nodes
            )
            weights = {
                node: self.calculate_transition_probability(
                    current_node, node, visited_nodes, agv, current_battery
                )
                for node in candidates
            }
            total_weight = sum(weights.values())

            if total_weight > 0.0:
                selected = random.random() * total_weight
                cumulative_weight = 0.0
                next_node = candidates[-1]
                for node in candidates:
                    cumulative_weight += weights[node]
                    if selected <= cumulative_weight:
                        next_node = node
                        break
            else:
                # This fallback keeps construction defined if all weights vanish.
                next_node = random.choice(candidates)

            route.append(next_node)
            visited_nodes.add(next_node)
            current_battery -= self.energy_consumption.get(
                (current_node, next_node), 0.0
            )

            station_ids = {station.id for station in self.charging_stations}
            if next_node in station_ids:
                current_battery = agv.battery_capacity
            else:
                task = next(
                    task
                    for task in remaining_tasks.values()
                    if next_node in (task.pickup, task.dropoff)
                )
                if next_node == task.pickup:
                    picked_up_tasks.add(task.id)
                else:
                    del remaining_tasks[task.id]

            current_node = next_node

        route.append("O")
        return route

    def evaluate_route(self, route, agv):
        """Sum travel time and count reserve violations on every travel leg."""
        if len(route) < 2:
            return 0.0, 0

        total_cost = 0.0
        current_battery = agv.initial_battery
        violations = 0
        station_ids = {station.id for station in self.charging_stations}

        for from_node, to_node in zip(route[:-1], route[1:]):
            total_cost += self.travel_times.get((from_node, to_node), 0.0)
            current_battery -= self.energy_consumption.get(
                (from_node, to_node), 0.0
            )

            # Check arrival reserve before charging can restore the battery.
            if current_battery < agv.battery_min:
                violations += 1
            if to_node in station_ids:
                current_battery = agv.battery_capacity

        return total_cost, violations

    def update_pheromones(self, ant_solutions):
        """Evaporate trails, then reinforce edges in feasible solutions."""
        for edge in self.pheromone:
            self.pheromone[edge] *= 1.0 - self.rho

        for solution in ant_solutions:
            if solution.feasible and solution.cost > 0.0:
                deposit = self.q / solution.cost
                for route in solution.routes.values():
                    for edge in zip(route[:-1], route[1:]):
                        if edge in self.pheromone:
                            self.pheromone[edge] += deposit

    def calculate_solution_diversity(self, ant_solutions):
        """Return mean pairwise route-set distance in the ant population."""
        if len(ant_solutions) < 2:
            return 0.0

        distances = []
        for i in range(len(ant_solutions)):
            for j in range(i + 1, len(ant_solutions)):
                distances.append(
                    self.calculate_solution_distance(
                        ant_solutions[i], ant_solutions[j]
                    )
                )
        return float(np.mean(distances))

    def calculate_solution_distance(self, sol1, sol2):
        """Compare solutions by the node sets used on corresponding AGVs."""
        if not self.agvs:
            return 0.0

        distance = 0.0
        for agv in self.agvs:
            route1 = sol1.routes.get(agv.id, [])
            route2 = sol2.routes.get(agv.id, [])
            set1 = set(route1)
            set2 = set(route2)
            denominator = max(len(set1), len(set2))
            if denominator:
                distance += len(set1.symmetric_difference(set2)) / denominator
        return distance / len(self.agvs)

    def optimize(self):
        """Run the ACO construction, evaluation, and pheromone-update loop."""
        print(
            f"Starting ACO with {self.num_ants} ants "
            f"for {self.generations} generations..."
        )

        for generation in range(self.generations):
            ant_solutions = [
                self.construct_ant_solution(ant_id)
                for ant_id in range(self.num_ants)
            ]
            feasible_solutions = [
                solution for solution in ant_solutions if solution.feasible
            ]

            if feasible_solutions:
                generation_best = min(
                    feasible_solutions, key=lambda solution: solution.cost
                )
                if generation_best.cost < self.best_cost:
                    self.best_solution = generation_best
                    self.best_cost = generation_best.cost
                average_cost = float(
                    np.mean([solution.cost for solution in feasible_solutions])
                )
            else:
                average_cost = float("nan")

            self.update_pheromones(ant_solutions)
            diversity = self.calculate_solution_diversity(ant_solutions)
            self.convergence_history.append(
                {
                    "generation": generation,
                    "best_cost": self.best_cost,
                    "avg_cost": average_cost,
                    "feasible_count": len(feasible_solutions),
                }
            )
            self.diversity_history.append(diversity)

            if (generation + 1) % 20 == 0:
                average_text = (
                    f"{average_cost:.2f}"
                    if np.isfinite(average_cost)
                    else "not available"
                )
                best_text = (
                    f"{self.best_cost:.2f}"
                    if self.best_solution is not None
                    else "not available"
                )
                print(
                    f"  Generation {generation + 1:3d}: "
                    f"best={best_text}, average feasible cost={average_text}, "
                    f"feasible={len(feasible_solutions)}/{self.num_ants}"
                )

        if self.best_solution is None:
            print("ACO completed without finding a feasible solution.")
        else:
            print(f"ACO completed. Best feasible cost: {self.best_cost:.2f}")

        return (
            self.best_solution,
            self.convergence_history,
            self.diversity_history,
        )

In [ ]:
aco = AGVAntColonyOptimizer(
    locations=locations,
    travel_times=travel_times,
    energy_consumption=energy_consumption,
    charging_stations=charging_stations,
    agvs=agvs,
    tasks=tasks,
    num_ants=20,
    generations=100,
    alpha=1.0,
    beta=2.0,
    rho=0.1,
    q=1.0,
)

start_time = time.time()
best_solution, convergence_history, diversity_history = aco.optimize()
computation_time = time.time() - start_time
print(f"\nACO run time: {computation_time:.2f} seconds")

print("\n" + "=" * 60)
print("BEST FEASIBLE ACO SOLUTION")
print("=" * 60)

if best_solution is not None:
    print(f"Total travel-time cost: {best_solution.cost:.2f}")
    print(f"Feasible: {best_solution.feasible}")
    print(f"Battery-reserve violations: {best_solution.battery_violations}")

    for agv_id, route in best_solution.routes.items():
        if len(route) > 2:
            agv = next(vehicle for vehicle in agvs if vehicle.id == agv_id)
            route_cost, violations = aco.evaluate_route(route, agv)
            print(f"\nAGV {agv_id}:")
            print(f"  Route: {' -> '.join(route)}")
            print(f"  Travel-time cost: {route_cost:.2f}")
            print(f"  Battery-reserve violations: {violations}")

    # Confirm complete task coverage and pickup-before-drop-off precedence.
    task_nodes = [node for route in best_solution.routes.values() for node in route]
    for task in tasks:
        assert task_nodes.count(task.pickup) == 1
        assert task_nodes.count(task.dropoff) == 1
        assigned_route = next(
            route
            for route in best_solution.routes.values()
            if task.pickup in route
        )
        assert assigned_route.index(task.pickup) < assigned_route.index(task.dropoff)

    # Confirm that the reported objective equals the sum of evaluated route costs.
    evaluated_total = sum(
        aco.evaluate_route(
            route, next(vehicle for vehicle in agvs if vehicle.id == agv_id)
        )[0]
        for agv_id, route in best_solution.routes.items()
    )
    assert np.isclose(evaluated_total, best_solution.cost)
else:
    print("No feasible solution was found in this run.")

### Reading the solution report

Each AGV route is an ordered sequence of locations. A task's pickup and drop-off appear in the same AGV route, with pickup first; charging stops, when selected, appear between route locations. The route cost is the sum of its travel times, and the reported battery violations count travel-leg arrivals below the AGV's minimum reserve, including the return trip. A feasible result has no such violations. The assertions in the code check task coverage, precedence, and agreement between the reported total cost and the evaluated route costs.

Because task assignments and route choices are stochastic, the best solution is the best feasible candidate found during this particular run. It is not necessarily the global optimum. If no feasible candidate is found, the report states that explicitly rather than presenting an infeasible route as a feasible result.

In [ ]:
def compare_with_heuristic(
    travel_times, energy_consumption, agvs, tasks, aco, best_solution
):
    """Compare ACO with a simple sequential greedy reference construction."""
    print("\n" + "=" * 60)
    print("COMPARISON WITH SIMPLE GREEDY REFERENCE")
    print("=" * 60)

    class SimpleHeuristic:
        """Append each task's pickup and drop-off to the least-cost AGV."""

        def __init__(self, travel_times, energy_consumption, agvs):
            self.travel_times = travel_times
            self.energy_consumption = energy_consumption
            self.agvs = agvs

        def solve(self, tasks):
            routes = {agv.id: ["O"] for agv in self.agvs}
            for task in tasks:
                best_agv = min(
                    self.agvs,
                    key=lambda agv: (
                        self.travel_times.get(("O", task.pickup), 0.0)
                        + self.travel_times.get(
                            (task.pickup, task.dropoff), 0.0
                        )
                        + self.travel_times.get((task.dropoff, "O"), 0.0)
                    ),
                )
                routes[best_agv.id].extend([task.pickup, task.dropoff])

            for route in routes.values():
                if len(route) > 1:
                    route.append("O")
            return routes

    heuristic = SimpleHeuristic(travel_times, energy_consumption, agvs)
    heuristic_routes = heuristic.solve(tasks)
    heuristic_cost = 0.0
    heuristic_violations = 0

    for agv_id, route in heuristic_routes.items():
        if len(route) > 2:
            agv = next(vehicle for vehicle in agvs if vehicle.id == agv_id)
            cost, violations = aco.evaluate_route(route, agv)
            heuristic_cost += cost
            heuristic_violations += violations

    print("Simple greedy reference:")
    print(f"  Total travel-time cost: {heuristic_cost:.2f}")
    print(f"  Battery-reserve violations: {heuristic_violations}")

    if best_solution is None:
        print("\nACO: no feasible solution was found.")
        aco_cost = float("nan")
        aco_violations = float("nan")
        cost_improvement = float("nan")
        violation_reduction = float("nan")
    else:
        aco_cost = best_solution.cost
        aco_violations = best_solution.battery_violations
        print("\nBest feasible ACO solution:")
        print(f"  Total travel-time cost: {aco_cost:.2f}")
        print(f"  Battery-reserve violations: {aco_violations}")

        cost_improvement = (
            100.0 * (heuristic_cost - aco_cost) / heuristic_cost
            if heuristic_cost > 0.0
            else 0.0
        )
        violation_reduction = heuristic_violations - aco_violations
        print(f"\nCost difference relative to reference: {cost_improvement:.1f}%")
        print(f"Battery-violation reduction: {violation_reduction}")

    return {
        "heuristic_cost": heuristic_cost,
        "aco_cost": aco_cost,
        "heuristic_violations": heuristic_violations,
        "aco_violations": aco_violations,
        "cost_improvement": cost_improvement,
        "violation_reduction": violation_reduction,
    }


comparison_results = compare_with_heuristic(
    travel_times, energy_consumption, agvs, tasks, aco, best_solution
)

### Interpreting the benchmark

The reference construction assigns each task to the AGV with the lowest isolated depot–pickup–drop-off–depot travel-time estimate, then appends tasks in their listed order. It does not optimize task order or insert charging stops. It is therefore a simple benchmark, not an implementation of a more elaborate insertion heuristic and not a proof that ACO is better. A positive cost difference means the ACO cost is lower than this reference cost; battery violations should be interpreted separately because the reference route may be infeasible.

In [ ]:
def visualize_aco_results(
    locations, agvs, best_solution, convergence_history,
    diversity_history, comparison_results
):
    """Plot routes, convergence, diversity, and benchmark metrics."""
    fig, axes = plt.subplots(2, 2, figsize=(16, 12))
    fig.suptitle("Ant Colony Optimization: Results", fontsize=14, fontweight="bold")

    # Terminal layout: marker shapes identify location types; line colors identify AGVs.
    ax1 = axes[0, 0]
    marker_specs = {
        "depot": ("ks", "Depot"),
        "pickup": ("g^", "Pickup"),
        "drop-off": ("r^", "Drop-off"),
        "charging": ("bs", "Charging station"),
    }
    legend_handles = {}
    for loc_id, loc in locations.items():
        if loc_id == "O":
            category = "depot"
        elif loc_id.startswith("P"):
            category = "pickup"
        elif loc_id.startswith("D"):
            category = "drop-off"
        else:
            category = "charging"
        marker, label = marker_specs[category]
        handle, = ax1.plot(
            loc.x, loc.y, marker, markersize=9, label=label
        )
        legend_handles.setdefault(label, handle)
        ax1.annotate(loc_id, (loc.x, loc.y), xytext=(3, 3),
                     textcoords="offset points", fontsize=8)

    route_colors = plt.get_cmap("tab10")
    if best_solution is not None:
        for index, (agv_id, route) in enumerate(best_solution.routes.items()):
            if len(route) < 2:
                continue
            color = route_colors(index % 10)
            for from_node, to_node in zip(route[:-1], route[1:]):
                from_loc = locations[from_node]
                to_loc = locations[to_node]
                is_charging_stop = to_node.startswith("C")
                handle, = ax1.plot(
                    [from_loc.x, to_loc.x],
                    [from_loc.y, to_loc.y],
                    color=color,
                    linestyle="--" if is_charging_stop else "-",
                    linewidth=1.8,
                    alpha=0.8,
                    label=f"AGV {agv_id}",
                )
                legend_handles.setdefault(f"AGV {agv_id}", handle)

    ax1.set_xlabel("X coordinate (distance units)")
    ax1.set_ylabel("Y coordinate (distance units)")
    ax1.set_title("Best Feasible ACO Routes (dashed: arrival at a charger)")
    ax1.grid(True, alpha=0.3)
    ax1.legend(legend_handles.values(), legend_handles.keys(),
               bbox_to_anchor=(1.02, 1), loc="upper left")

    # Best-so-far cost and feasible-population statistics by generation.
    ax2 = axes[0, 1]
    generations = [item["generation"] for item in convergence_history]
    best_costs = [
        item["best_cost"] if np.isfinite(item["best_cost"]) else np.nan
        for item in convergence_history
    ]
    average_costs = [item["avg_cost"] for item in convergence_history]
    feasible_counts = [item["feasible_count"] for item in convergence_history]

    ax2_twin = ax2.twinx()
    line_best, = ax2.plot(generations, best_costs, "b-", linewidth=2,
                          label="Best feasible cost so far")
    line_average, = ax2.plot(generations, average_costs, "g--", linewidth=1.5,
                             label="Mean feasible cost")
    line_feasible, = ax2_twin.plot(
        generations, feasible_counts, "r-", linewidth=1.5,
        alpha=0.75, label="Feasible ants"
    )
    ax2.set_xlabel("Generation")
    ax2.set_ylabel("Travel-time cost")
    ax2_twin.set_ylabel("Feasible ants per generation")
    ax2.set_title("Feasibility and Cost Across Generations")
    ax2.grid(True, alpha=0.3)
    ax2.legend(
        [line_best, line_average, line_feasible],
        [line.get_label() for line in (line_best, line_average, line_feasible)],
        loc="best",
    )

    # Diversity is based on differences between the sets of nodes used per AGV.
    ax3 = axes[1, 0]
    ax3.plot(range(len(diversity_history)), diversity_history,
             color="purple", linewidth=1.8)
    ax3.set_xlabel("Generation")
    ax3.set_ylabel("Mean pairwise route-set distance")
    ax3.set_title("Population Diversity")
    ax3.grid(True, alpha=0.3)

    # Cost and violation measures use separate axes because they have different units.
    ax4 = axes[1, 1]
    methods = ["Greedy reference", "ACO"]
    costs = [comparison_results["heuristic_cost"], comparison_results["aco_cost"]]
    violations = [
        comparison_results["heuristic_violations"],
        comparison_results["aco_violations"],
    ]
    positions = np.arange(len(methods))
    width = 0.35
    bars_cost = ax4.bar(
        positions - width / 2, costs, width,
        label="Travel-time cost", color="lightblue"
    )
    ax4_twin = ax4.twinx()
    bars_violations = ax4_twin.bar(
        positions + width / 2, violations, width,
        label="Battery-reserve violations", color="lightcoral"
    )
    ax4.set_xlabel("Construction method")
    ax4.set_ylabel("Travel-time cost")
    ax4_twin.set_ylabel("Battery-reserve violations")
    ax4.set_title("ACO and Greedy Reference")
    ax4.set_xticks(positions)
    ax4.set_xticklabels(methods)
    ax4.legend([bars_cost], ["Travel-time cost"], loc="upper left")
    ax4_twin.legend([bars_violations], ["Battery-reserve violations"],
                    loc="upper right")

    for bars, axis in ((bars_cost, ax4), (bars_violations, ax4_twin)):
        for bar in bars:
            height = bar.get_height()
            if np.isfinite(height):
                axis.annotate(
                    f"{height:.1f}",
                    (bar.get_x() + bar.get_width() / 2, height),
                    xytext=(0, 3),
                    textcoords="offset points",
                    ha="center",
                    va="bottom",
                )

    improvement = comparison_results["cost_improvement"]
    if np.isfinite(improvement):
        ax4.text(
            0.5, 0.96, f"Cost difference: {improvement:.1f}%",
            transform=ax4.transAxes, ha="center", va="top",
            bbox={"boxstyle": "round", "facecolor": "wheat", "alpha": 0.5},
        )

    fig.tight_layout()
    plt.show()
    return fig


fig = visualize_aco_results(
    locations, agvs, best_solution, convergence_history,
    diversity_history, comparison_results
)

### Reading the plots

The route panel uses line color to identify the AGV and line style to distinguish a leg arriving at a charging station. The convergence panel reports the best feasible cost found up to each generation and the mean cost among feasible ants in that generation; a generation with no feasible ants has no mean-cost point. The feasible-ant count is shown on a separate axis. Diversity is measured using the symmetric difference between the sets of nodes assigned to each AGV, averaged across pairs of ant solutions. It measures assignment and node-set variation, not route-order differences when two routes visit the same nodes. In the benchmark panel, cost and battery-violation counts use separate axes because they are different quantities.

In [ ]:
def analyze_pheromone_trails(aco):
    """Report and plot pheromone levels learned by the ACO run."""
    print("\n" + "=" * 60)
    print("PHEROMONE TRAIL ANALYSIS")
    print("=" * 60)

    sorted_trails = sorted(
        aco.pheromone.items(), key=lambda item: item[1], reverse=True
    )
    print("Top 10 directed pheromone trails:")
    for index, (edge, value) in enumerate(sorted_trails[:10], start=1):
        print(f"  {index:2d}. {edge[0]} -> {edge[1]}: {value:.4f}")

    pheromone_values = np.asarray(list(aco.pheromone.values()), dtype=float)
    print("\nPheromone statistics:")
    print(f"  Mean: {np.mean(pheromone_values):.4f}")
    print(f"  Standard deviation: {np.std(pheromone_values):.4f}")
    print(f"  Minimum: {np.min(pheromone_values):.4f}")
    print(f"  Maximum: {np.max(pheromone_values):.4f}")

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))
    ax1.hist(
        pheromone_values, bins=20, color="skyblue", alpha=0.7,
        edgecolor="black"
    )
    ax1.set_xlabel("Pheromone level")
    ax1.set_ylabel("Number of directed edges")
    ax1.set_title("Pheromone Trail Distribution")
    ax1.grid(True, alpha=0.3)

    key_nodes = ["O", "P1", "P2", "P3", "D1", "D2", "D3", "C1", "C2"]
    pheromone_matrix = np.zeros((len(key_nodes), len(key_nodes)))
    for i, node1 in enumerate(key_nodes):
        for j, node2 in enumerate(key_nodes):
            if i != j:
                pheromone_matrix[i, j] = aco.pheromone.get((node1, node2), 0.0)

    image = ax2.imshow(pheromone_matrix, cmap="YlOrRd", aspect="auto")
    ax2.set_xticks(range(len(key_nodes)))
    ax2.set_xticklabels(key_nodes, rotation=45)
    ax2.set_yticks(range(len(key_nodes)))
    ax2.set_yticklabels(key_nodes)
    ax2.set_xlabel("Destination node")
    ax2.set_ylabel("Origin node")
    ax2.set_title("Directed Pheromone Levels for Selected Nodes")
    colorbar = plt.colorbar(image, ax=ax2)
    colorbar.set_label("Pheromone level")

    fig.tight_layout()
    plt.show()
    return sorted_trails[:10]


top_trails = analyze_pheromone_trails(aco)

## Summary and interpretation

### What the algorithm does

ACO repeatedly constructs candidate routes and shares information through pheromone trails. Pheromone reinforces edges used by feasible solutions, with larger deposits from lower-cost solutions. Evaporation reduces the influence of accumulated trails over time. The heuristic term favors locally attractive edges, while probabilistic selection preserves the possibility of choosing other eligible edges. Battery-aware weighting guides ants toward charging stations when a move threatens the reserve, and route evaluation determines whether the completed solution is feasible.

### What the results can establish

The best cost, feasible-ant counts, diversity curve, route plot, and pheromone distribution describe the run produced by the notebook's data, parameters, and random seed. They do not establish a percentage improvement over other instances, prove convergence to a global optimum, or guarantee a particular number of generations will be sufficient. The greedy reference is a single, deliberately simple construction; its comparison is specific to this instance and must be considered together with its battery violations.

The diversity metric compares node sets for each AGV. It can reveal differences in task assignment and charging-node use, but cannot distinguish two routes that use the same node set in different orders. Strong pheromone trails indicate that feasible ants repeatedly used those directed edges and that their solutions contributed deposits; a strong trail alone does not prove that an edge belongs to a globally optimal route.

### Model limitations and natural extensions

The model uses Euclidean travel, a common travel-time scale, instantaneous full recharging, and a battery reserve checked at the end of each travel leg. It does not model congestion, time windows, deadlines, service times, AGV-specific travel speeds, charging queues, partial charging, or station capacity. Those factors can be incorporated as additional model data and feasibility conditions while retaining the ACO construction-and-pheromone-update framework. Any such extension requires revising route evaluation and the interpretation of cost and feasibility consistently.